This is a prototype, not a real billing tool. ICD-10-CM diagnosis coding should follow official ICD-10-CM guidelines, and CPT is maintained by the AMA for reporting medical services and procedures. Real coding requires current official resources and qualified human review.

Clinical Note to Candidate ICD/CPT Coding Pipeline

In this notebook, we will build a beginner-friendly system that takes raw clinical note text and creates a possible coded output.

The workflow is:

Clinical Note
↓
Tokenization + Normalization
↓
NER, find entities
↓
Context + Time understanding
↓
Classification, predict coding concepts
↓
Map to ICD/CPT
↓
Final coded output

Important:

This notebook is for learning only.

It does not replace a certified medical coder.

It does not produce final billable codes.

It creates candidate codes that need human review.

What This Project Will Do

We will use fake clinical notes.

For each note, the system will:

Clean the note
Split the note into tokens
Normalize abbreviations
Find medical entities
Find time expressions
Classify possible coding concepts
Map concepts to candidate ICD/CPT codes
Validate evidence
Print final coder-friendly output

In [1]:
!pip install pandas transformers torch python-dateutil

Step 1: Import Libraries

We need:

pandas: to store clinical notes in a table
re: to clean text using patterns
transformers: to load a biomedical NER model
dateutil: to handle relative dates like “3 months ago”

In [2]:
import pandas as pd
import re

from transformers import pipeline
from dateutil.relativedelta import relativedelta
from datetime import datetime

Step 2: Create Raw Clinical Notes

These notes are fake toy examples.

Do not use real patient data in a practice notebook.

Each note has:

note_id
note_date
raw_note_text

The note_date is important because time phrases like “today” or “3 months ago” need a reference date.

In [3]:
clinical_notes = [
    {
        "note_id": 1,
        "note_date": "2024-01-15",
        "raw_note_text": """
        Pt is a 58 y/o male with Type 2 Diabetes.
        Currently taking Metformin 1000mg BID.
        HbA1c today is 7.8%, previously 7.2% three months ago.
        Denies chest pain.
        Plan: continue Metformin and follow up in 3 months.
        """
    },
    {
        "note_id": 2,
        "note_date": "2024-01-20",
        "raw_note_text": """
        Patient has HTN treated with Lisinopril 20mg daily.
        BP today 145/92.
        Reports dizziness last week after standing.
        Plan: monitor BP at home and return in 2 weeks.
        """
    },
    {
        "note_id": 3,
        "note_date": "2024-01-22",
        "raw_note_text": """
        Patient presents to ED with chest pain and shortness of breath.
        EKG shows ST elevation.
        Aspirin 325mg given in ED.
        Cardiology consulted immediately.
        """
    }
]

df = pd.DataFrame(clinical_notes)

df

,note_id,note_date,raw_note_text
0,1,2024-01-15,\n Pt is a 58 y/o male with Type 2 Diab...
1,2,2024-01-20,\n Patient has HTN treated with Lisinop...
2,3,2024-01-22,\n Patient presents to ED with chest pa...


Step 3: Clean the Raw Text

Clinical notes are often messy.

They may contain:

extra line breaks
repeated spaces
abbreviations
medication doses
lab values

We will clean only the spacing.

We do not want to remove useful clinical details like:

1000mg
145/92
HbA1c
BID
y/o
EKG

In [4]:
def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

In [5]:
df["clean_text"] = df["raw_note_text"].apply(clean_text)

df[["note_id", "clean_text"]]

,note_id,clean_text
0,1,Pt is a 58 y/o male with Type 2 Diabetes. Curr...
1,2,Patient has HTN treated with Lisinopril 20mg d...
2,3,Patient presents to ED with chest pain and sho...


Explanation

This line:

re.sub(r"\s+", " ", text)

means:

Find extra spaces, tabs, and line breaks.

Replace them with one normal space.

This line:

text.strip()

removes spaces from the beginning and end.

Step 4: Tokenization

Tokenization means splitting text into smaller pieces.

Example:

Patient has diabetes.

Tokens:

Patient
has
diabetes

We will use simple regex tokenization so beginners do not need extra NLTK setup.

In [6]:
def tokenize_text(text):
    tokens = re.findall(r"[A-Za-z0-9/%\.]+", text)
    return token

In [7]:
df["tokens"] = df["clean_text"].apply(tokenize_text)

df[["note_id", "tokens"]]

,note_id,tokens
0,1,"[Pt, is, a, 58, y/o, male, with, Type, 2, Diab..."
1,2,"[Patient, has, HTN, treated, with, Lisinopril,..."
2,3,"[Patient, presents, to, ED, with, chest, pain,..."


Step 5: Normalization

Normalization means converting clinical abbreviations into clearer words.

Examples:

HTN → hypertension
BP → blood pressure
EKG → electrocardiogram
SOB → shortness of breath
BID → twice daily
Pt → patient

Why do we normalize?

Because models and rules work better when the language is clearer.

In [8]:
abbreviation_map = {
    "pt": "patient",
    "htn": "hypertension",
    "bp": "blood pressure",
    "ekg": "electrocardiogram",
    "sob": "shortness of breath",
    "bid": "twice daily",
    "ed": "emergency department"
}


def normalize_text(text):
    normalized = text.lower()

    for abbreviation, full_form in abbreviation_map.items():
        pattern = r"\b" + abbreviation + r"\b"
        normalized = re.sub(pattern, full_form, normalized)

    normalized = re.sub(r"\s+", " ", normalized).strip()

    return normalized

In [9]:
df["normalized_text"] = df["clean_text"].apply(normalize_text)

df[["note_id", "clean_text", "normalized_text"]]

,note_id,clean_text,normalized_text
0,1,Pt is a 58 y/o male with Type 2 Diabetes. Curr...,patient is a 58 y/o male with type 2 diabetes....
1,2,Patient has HTN treated with Lisinopril 20mg d...,patient has hypertension treated with lisinopr...
2,3,Patient presents to ED with chest pain and sho...,patient presents to emergency department with ...


This dictionary:

abbreviation_map

stores abbreviations and their meanings.

Example:

"htn": "hypertension"

This loop:

for abbreviation, full_form in abbreviation_map.items():

goes through every abbreviation and replaces it with the full meaning.

The pattern:

\b

means word boundary.

It helps replace the abbreviation only when it appears as a full word.

Step 6: Load Biomedical NER Model

NER means Named Entity Recognition.

NER finds medical words or phrases.

Examples:

Type 2 Diabetes
Metformin
HbA1c
chest pain
Lisinopril
Aspirin

We will use a biomedical NER model from Hugging Face.

In [10]:
ner_model = pipeline(
    task="token-classification",
    model="d4data/biomedical-ner-all",
    aggregation_strategy="simple"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/266M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Step 7: Extract Entities

The model output can look complicated.

So we will keep only:

text
label
confidence

In [11]:
def extract_entities(text):
    raw_entities = ner_model(text)

    entities = []

    for entity in raw_entities:
        entities.append({
            "text": entity["word"],
            "label": entity["entity_group"],
            "confidence": round(entity["score"], 2)
        })

    return entities

In [12]:
df["entities"] = df["clean_text"].apply(extract_entities)

df[["note_id", "entities"]]

,note_id,entities
0,1,"[{'text': 'pt', 'label': 'Disease_disorder', '..."
1,2,"[{'text': 'htn', 'label': 'Medication', 'confi..."
2,3,"[{'text': 'presents', 'label': 'Clinical_event..."


In [13]:
for i in range(len(df)):
    print("NOTE ID:", df.iloc[i]["note_id"])
    print("NOTE:", df.iloc[i]["clean_text"])
    print()
    print("ENTITIES FOUND:")

    for entity in df.iloc[i]["entities"]:
        print("Entity:", entity["text"])
        print("Label:", entity["label"])
        print("Confidence:", entity["confidence"])
        print()

    print("=" * 80)

NOTE ID: 1
NOTE: Pt is a 58 y/o male with Type 2 Diabetes. Currently taking Metformin 1000mg BID. HbA1c today is 7.8%, previously 7.2% three months ago. Denies chest pain. Plan: continue Metformin and follow up in 3 months.

ENTITIES FOUND:
Entity: pt
Label: Disease_disorder
Confidence: 0.98

Entity: 58 y / o
Label: Lab_value
Confidence: 1.0

Entity: male
Label: Sex
Confidence: 0.99

Entity: type 2
Label: History
Confidence: 0.99

Entity: metform
Label: Medication
Confidence: 0.95

Entity: 1000mg
Label: Dosage
Confidence: 0.78

Entity: hba1c
Label: Medication
Confidence: 0.98

Entity: 7
Label: Lab_value
Confidence: 0.95

Entity: 7. 2 %
Label: Lab_value
Confidence: 0.89

Entity: three months
Label: Duration
Confidence: 0.61

Entity: chest
Label: Biological_structure
Confidence: 1.0

Entity: metform
Label: Medication
Confidence: 0.92

NOTE ID: 2
NOTE: Patient has HTN treated with Lisinopril 20mg daily. BP today 145/92. Reports dizziness last week after standing. Plan: monitor BP at home 

Step 8: Context + Time Understanding

Now we find time expressions.

Examples:

today
three months ago
last week
in 3 months
in 2 weeks
immediately

Why do we need this?

Because coding and clinical understanding often depend on whether something is:

current
historical
planned
acute
follow-up
Cell 26 — Code

In [14]:
def normalize_relative_time(time_text, note_date):
    note_datetime = datetime.strptime(note_date, "%Y-%m-%d")

    time_text_lower = time_text.lower()

    if time_text_lower == "today":
        return note_datetime.strftime("%Y-%m-%d")

    if time_text_lower in ["three months ago", "3 months ago"]:
        estimated_date = note_datetime - relativedelta(months=3)
        return estimated_date.strftime("%Y-%m-%d")

    if time_text_lower == "last week":
        estimated_date = note_datetime - relativedelta(weeks=1)
        return estimated_date.strftime("%Y-%m-%d")

    if time_text_lower == "in 3 months":
        estimated_date = note_datetime + relativedelta(months=3)
        return estimated_date.strftime("%Y-%m-%d")

    if time_text_lower == "in 2 weeks":
        estimated_date = note_datetime + relativedelta(weeks=2)
        return estimated_date.strftime("%Y-%m-%d")

    if time_text_lower == "immediately":
        return note_datetime.strftime("%Y-%m-%d") + " same day / immediate"

    return "not stated"

Simplified Human Version

The code is basically saying:

If text says "today"
    return today's date

If text says "3 months ago"
    subtract 3 months

If text says "last week"
    subtract 1 week

If text says "in 3 months"
    add 3 months

Otherwise
    return "not stated"
Most Important Concepts You Just Learned
1. Functions
def something():

Reusable tool.

2. Conditions
if something:

Decision making.

3. String methods
.lower()

Modify text.

4. Date conversion
string → datetime → string

Very common.

5. Date arithmetic
+ relativedelta()
- relativedelta()

Move through time.

The REAL Thing to Understand

Don’t memorize:

strftime
strptime
relativedelta

Instead understand the flow:

Receive text
↓
Convert date
↓
Check condition
↓
Calculate date
↓
Return result

In [15]:
def extract_time_expressions(text, note_date):
    possible_times = [
        "today",
        "three months ago",
        "3 months ago",
        "last week",
        "in 3 months",
        "in 2 weeks",
        "immediately"
    ]

    time_results = []

    text_lower = text.lower()

    for time_text in possible_times:
        if time_text in text_lower:
            time_results.append({
                "time_text": time_text,
                "normalized_time": normalize_relative_time(time_text, note_date)
            })

    return time_results

In [16]:
df["time_expressions"] = df.apply(
    lambda row: extract_time_expressions(
        row["clean_text"],
        row["note_date"]
    ),
    axis=1
)

df[["note_id", "note_date", "time_expressions"]]

,note_id,note_date,time_expressions
0,1,2024-01-15,"[{'time_text': 'today', 'normalized_time': '20..."
1,2,2024-01-20,"[{'time_text': 'today', 'normalized_time': '20..."
2,3,2024-01-22,"[{'time_text': 'immediately', 'normalized_time..."


This function looks for time words.

If it finds:

today

and the note date is:

2024-01-15

then normalized_time becomes:

2024-01-15

If it finds:

three months ago

then it subtracts 3 months from the note date.

This is not perfect, but it is a good beginner version.

Step 9: Classification, Predict Coding Concepts

Before mapping to ICD/CPT, we first classify the note into coding concepts.

This means:

The note mentions diabetes → predict concept type_2_diabetes
The note mentions hypertension → predict concept hypertension
The note mentions chest pain → predict concept chest_pain
The note mentions EKG → predict concept ekg_performed_or_reviewed

Important:

This is a simple rule-based classifier for learning.

In real systems, this could be replaced by:

a fine-tuned ClinicalBERT classifier
a Clinical LLM
a professional coding engine
human coder review

In [17]:
def classify_coding_concepts(text):
    text_lower = text.lower()

    concepts = []

    if "type 2 diabetes" in text_lower or "diabetes" in text_lower:
        concepts.append({
            "concept": "type_2_diabetes",
            "category": "diagnosis",
            "evidence": "Type 2 Diabetes" if "type 2 diabetes" in text_lower else "diabetes"
        })

    if "hypertension" in text_lower:
        concepts.append({
            "concept": "hypertension",
            "category": "diagnosis",
            "evidence": "hypertension"
        })

    if "chest pain" in text_lower and "denies chest pain" not in text_lower:
        concepts.append({
            "concept": "chest_pain",
            "category": "diagnosis_or_symptom",
            "evidence": "chest pain"
        })

    if "shortness of breath" in text_lower:
        concepts.append({
            "concept": "shortness_of_breath",
            "category": "diagnosis_or_symptom",
            "evidence": "shortness of breath"
        })

    if "st elevation" in text_lower or "electrocardiogram shows" in text_lower:
        concepts.append({
            "concept": "abnormal_ekg",
            "category": "diagnosis_or_finding",
            "evidence": "ST elevation" if "st elevation" in text_lower else "electrocardiogram shows"
        })

    if "electrocardiogram" in text_lower:
        concepts.append({
            "concept": "ekg_performed_or_reviewed",
            "category": "procedure_or_test",
            "evidence": "EKG" if "ekg" in text_lower else "electrocardiogram"
        })

    if "follow up" in text_lower or "return in" in text_lower:
        concepts.append({
            "concept": "follow_up_planned",
            "category": "care_plan",
            "evidence": "follow up" if "follow up" in text_lower else "return in"
        })

    if "aspirin" in text_lower and "given" in text_lower:
        concepts.append({
            "concept": "medication_given",
            "category": "treatment",
            "evidence": "Aspirin 325mg given" if "aspirin 325mg given" in text_lower else "aspirin given"
        })

    return concepts

In [18]:
df["coding_concepts"] = df["normalized_text"].apply(classify_coding_concepts)

df[["note_id", "coding_concepts"]]

,note_id,coding_concepts
0,1,"[{'concept': 'type_2_diabetes', 'category': 'd..."
1,2,"[{'concept': 'hypertension', 'category': 'diag..."
2,3,"[{'concept': 'chest_pain', 'category': 'diagno..."


This is the basic pattern:

if "diabetes" in note:
predict diabetes concept

The classifier does not directly assign ICD/CPT codes yet.

It first predicts a concept.

Then the next step maps the concept to candidate codes.

This keeps the workflow easier to understand:

Text
↓
Concept
↓
Code

Step 10: Map Concepts to ICD/CPT

Now we create a small educational mapping.

Important:

This is a tiny toy mapping.

It is not a complete coding system.

Real coding must use official, current ICD/CPT resources and human review.

ICD-10-CM is used for diagnosis coding guidance in the United States, while CPT codes describe medical services and procedures performed by physicians and other qualified health care professionals.

In [19]:
concept_to_code_map = {
    "type_2_diabetes": [
        {
            "code_system": "ICD-10-CM",
            "code": "E11.9",
            "description": "Type 2 diabetes mellitus without complications",
            "coding_note": "Educational candidate only. Confirm complications and documentation."
        }
    ],
    "hypertension": [
        {
            "code_system": "ICD-10-CM",
            "code": "I10",
            "description": "Essential hypertension",
            "coding_note": "Educational candidate only. Confirm diagnosis documentation."
        }
    ],
    "chest_pain": [
        {
            "code_system": "ICD-10-CM",
            "code": "R07.9",
            "description": "Chest pain, unspecified",
            "coding_note": "Educational candidate only. Final coding depends on final diagnosis."
        }
    ],
    "shortness_of_breath": [
        {
            "code_system": "ICD-10-CM",
            "code": "R06.02",
            "description": "Shortness of breath",
            "coding_note": "Educational candidate only."
        }
    ],
    "abnormal_ekg": [
        {
            "code_system": "ICD-10-CM",
            "code": "R94.31",
            "description": "Abnormal electrocardiogram",
            "coding_note": "Educational candidate only. Confirm final interpretation."
        }
    ],
    "ekg_performed_or_reviewed": [
        {
            "code_system": "CPT",
            "code": "EKG CPT candidate",
            "description": "Electrocardiogram service candidate",
            "coding_note": "Exact CPT depends on tracing, interpretation, report, setting, and payer rules."
        }
    ],
    "follow_up_planned": [
        {
            "code_system": "Workflow",
            "code": "FOLLOW_UP",
            "description": "Follow-up planned",
            "coding_note": "Workflow action, not a billing code."
        }
    ],
    "medication_given": [
        {
            "code_system": "Workflow",
            "code": "MED_ADMIN_REVIEW",
            "description": "Medication administration mentioned",
            "coding_note": "Workflow flag. Procedure coding depends on setting and documentation."
        }
    ]
}

concept_to_code_map
│
├── "type_2_diabetes"
│        ↓
│      List
│        ↓
│      Dictionary
│         ├── code_system
│         ├── code
│         ├── description
│         └── coding_note
│
├── "hypertension"
│
└── "chest_pain"

Because one concept may eventually have:

multiple ICD codes
multiple CPT codes
multiple workflows

So list allows MANY coding options.

PART 3 — Inner Dictionary

This part:

{
    "code_system": "ICD-10-CM",
    "code": "E11.9",
    "description": "...",
    "coding_note": "..."
}

stores structured metadata.

Meaning of Each Field
code_system
"ICD-10-CM"

Which coding standard is being used.

code
"E11.9"

Why CPT Is Different

ICD codes usually describe diagnoses, symptoms, or findings.

CPT codes usually describe services and procedures.

For example:

diagnosis: diabetes
symptom: chest pain
procedure/service: EKG

CPT coding often needs more documentation than a short note gives.

For example, an EKG code may depend on:

whether tracing was performed
whether interpretation was performed
whether a formal report exists
care setting
payer rules

In [21]:
def map_concepts_to_codes(concepts):
    code_candidates = []

    for concept in concepts:
        concept_name = concept["concept"]

        if concept_name in concept_to_code_map:
            mapped_codes = concept_to_code_map[concept_name]

            for mapped_code in mapped_codes:
                code_candidate = {
                    "concept": concept_name,
                    "category": concept["category"],
                    "evidence": concept["evidence"],
                    "code_system": mapped_code["code_system"],
                    "code": mapped_code["code"],
                    "description": mapped_code["description"],
                    "coding_note": mapped_code["coding_note"],
                    "needs_coder_review": True
                }

                code_candidates.append(code_candidate)

    return code_candidates

In [22]:
df["code_candidates"] = df["coding_concepts"].apply(map_concepts_to_codes)

df[["note_id", "code_candidates"]]

,note_id,code_candidates
0,1,"[{'concept': 'type_2_diabetes', 'category': 'd..."
1,2,"[{'concept': 'hypertension', 'category': 'diag..."
2,3,"[{'concept': 'chest_pain', 'category': 'diagno..."


Step 11: Validate Evidence

Now we check whether the evidence appears in the note.

If the evidence is found in the note:

evidence_found_in_note = True

If not:

evidence_found_in_note = False
needs_coder_review = True

This helps reduce hallucination or unsupported coding suggestions.

In [23]:
def validate_code_evidence(note_text, code_candidates):
    note_lower = note_text.lower()

    validated_codes = []

    for candidate in code_candidates:
        evidence_lower = candidate["evidence"].lower()

        if evidence_lower in note_lower:
            candidate["evidence_found_in_note"] = True
            candidate["validation_reason"] = "Evidence found in note."
        else:
            candidate["evidence_found_in_note"] = False
            candidate["validation_reason"] = "Evidence not found exactly in note. Human review required."
            candidate["needs_coder_review"] = True

        validated_codes.append(candidate)

    return validated_codes

In [24]:
df["validated_codes"] = df.apply(
    lambda row: validate_code_evidence(
        row["clean_text"],
        row["code_candidates"]
    ),
    axis=1
)

df[["note_id", "validated_codes"]]

,note_id,validated_codes
0,1,"[{'concept': 'type_2_diabetes', 'category': 'd..."
1,2,"[{'concept': 'hypertension', 'category': 'diag..."
2,3,"[{'concept': 'chest_pain', 'category': 'diagno..."


Step 12: Final Coded Output

Now we combine everything:

clean note
tokens
normalized text
entities
time expressions
predicted concepts
candidate codes
validation result

This is the final output for a coder or doctor to review.

In [25]:
def print_final_coded_output(row):
    print("NOTE ID:", row["note_id"])
    print("NOTE DATE:", row["note_date"])
    print()

    print("CLEAN NOTE:")
    print(row["clean_text"])
    print()

    print("TOKENS:")
    print(row["tokens"])
    print()

    print("NORMALIZED TEXT:")
    print(row["normalized_text"])
    print()

    print("MEDICAL ENTITIES FOUND:")
    for entity in row["entities"]:
        print("-", entity["text"], "|", entity["label"], "| confidence:", entity["confidence"])
    print()

    print("TIME EXPRESSIONS:")
    for time_item in row["time_expressions"]:
        print("-", time_item["time_text"], "→", time_item["normalized_time"])
    print()

    print("PREDICTED CODING CONCEPTS:")
    for concept in row["coding_concepts"]:
        print("-", concept["concept"])
        print("  Category:", concept["category"])
        print("  Evidence:", concept["evidence"])
    print()

    print("FINAL CODE CANDIDATES:")
    for code in row["validated_codes"]:
        print()
        print("Concept:", code["concept"])
        print("Code system:", code["code_system"])
        print("Candidate code:", code["code"])
        print("Description:", code["description"])
        print("Evidence:", code["evidence"])
        print("Evidence found in note:", code["evidence_found_in_note"])
        print("Needs coder review:", code["needs_coder_review"])
        print("Validation:", code["validation_reason"])
        print("Coding note:", code["coding_note"])

    print()
    print("=" * 100)

In [26]:
for i in range(len(df)):
    print_final_coded_output(df.iloc[i])

NOTE ID: 1
NOTE DATE: 2024-01-15

CLEAN NOTE:
Pt is a 58 y/o male with Type 2 Diabetes. Currently taking Metformin 1000mg BID. HbA1c today is 7.8%, previously 7.2% three months ago. Denies chest pain. Plan: continue Metformin and follow up in 3 months.

TOKENS:
['Pt', 'is', 'a', '58', 'y/o', 'male', 'with', 'Type', '2', 'Diabetes.', 'Currently', 'taking', 'Metformin', '1000mg', 'BID.', 'HbA1c', 'today', 'is', '7.8%', 'previously', '7.2%', 'three', 'months', 'ago.', 'Denies', 'chest', 'pain.', 'Plan', 'continue', 'Metformin', 'and', 'follow', 'up', 'in', '3', 'months.']

NORMALIZED TEXT:
patient is a 58 y/o male with type 2 diabetes. currently taking metformin 1000mg twice daily. hba1c today is 7.8%, previously 7.2% three months ago. denies chest pain. plan: continue metformin and follow up in 3 months.

MEDICAL ENTITIES FOUND:
- pt | Disease_disorder | confidence: 0.98
- 58 y / o | Lab_value | confidence: 1.0
- male | Sex | confidence: 0.99
- type 2 | History | confidence: 0.99
- metfo

Step 13: Create a Compact Final Table

Sometimes you want the output in a table.

We will create one row per candidate code.

In [27]:
final_rows = []

for i in range(len(df)):
    row = df.iloc[i]

    for code in row["validated_codes"]:
        final_rows.append({
            "note_id": row["note_id"],
            "note_date": row["note_date"],
            "concept": code["concept"],
            "code_system": code["code_system"],
            "candidate_code": code["code"],
            "description": code["description"],
            "evidence": code["evidence"],
            "evidence_found_in_note": code["evidence_found_in_note"],
            "needs_coder_review": code["needs_coder_review"],
            "coding_note": code["coding_note"]
        })

final_code_table = pd.DataFrame(final_rows)

final_code_table

,note_id,note_date,concept,code_system,candidate_code,description,evidence,evidence_found_in_note,needs_coder_review,coding_note
0,1,2024-01-15,type_2_diabetes,ICD-10-CM,E11.9,Type 2 diabetes mellitus without complications,Type 2 Diabetes,True,True,Educational candidate only. Confirm complicati...
1,1,2024-01-15,follow_up_planned,Workflow,FOLLOW_UP,Follow-up planned,follow up,True,True,"Workflow action, not a billing code."
2,2,2024-01-20,hypertension,ICD-10-CM,I10,Essential hypertension,hypertension,False,True,Educational candidate only. Confirm diagnosis ...
3,2,2024-01-20,follow_up_planned,Workflow,FOLLOW_UP,Follow-up planned,return in,True,True,"Workflow action, not a billing code."
4,3,2024-01-22,chest_pain,ICD-10-CM,R07.9,"Chest pain, unspecified",chest pain,True,True,Educational candidate only. Final coding depen...
5,3,2024-01-22,shortness_of_breath,ICD-10-CM,R06.02,Shortness of breath,shortness of breath,True,True,Educational candidate only.
6,3,2024-01-22,abnormal_ekg,ICD-10-CM,R94.31,Abnormal electrocardiogram,ST elevation,True,True,Educational candidate only. Confirm final inte...
7,3,2024-01-22,ekg_performed_or_reviewed,CPT,EKG CPT candidate,Electrocardiogram service candidate,electrocardiogram,False,True,"Exact CPT depends on tracing, interpretation, ..."
8,3,2024-01-22,medication_given,Workflow,MED_ADMIN_REVIEW,Medication administration mentioned,Aspirin 325mg given,True,True,Workflow flag. Procedure coding depends on set...


What the Final Table Means

Each row is one possible coding suggestion.

Columns:

note_id
Which note the code came from.

concept
The clinical concept predicted by the classifier.

code_system
ICD-10-CM, CPT, or Workflow.

candidate_code
The possible code or workflow flag.

description
Plain-English meaning.

evidence
Text that supports the code.

evidence_found_in_note
Whether the evidence was found in the note.

needs_coder_review
Always True in this prototype because coding needs human review.

coding_note
Warning or explanation for coders.

Full Pipeline Function

Now we create one function that runs the whole workflow on a new clinical note.

This is useful because later you can give it any new note.

In [28]:
def clinical_note_to_candidate_codes(raw_note_text, note_date):
    clean = clean_text(raw_note_text)
    tokens = tokenize_text(clean)
    normalized = normalize_text(clean)
    entities = extract_entities(clean)
    times = extract_time_expressions(clean, note_date)
    concepts = classify_coding_concepts(normalized)
    code_candidates = map_concepts_to_codes(concepts)
    validated_codes = validate_code_evidence(clean, code_candidates)

    result = {
        "note_date": note_date,
        "clean_text": clean,
        "tokens": tokens,
        "normalized_text": normalized,
        "entities": entities,
        "time_expressions": times,
        "coding_concepts": concepts,
        "validated_codes": validated_codes
    }

    return result

In [29]:
new_note = """
Pt with Type 2 Diabetes on Metformin.
HbA1c today 8.1%.
No chest pain.
Return in 2 weeks.
"""

new_result = clinical_note_to_candidate_codes(
    raw_note_text=new_note,
    note_date="2024-02-01"
)

new_result

{'note_date': '2024-02-01',
 'clean_text': 'Pt with Type 2 Diabetes on Metformin. HbA1c today 8.1%. No chest pain. Return in 2 weeks.',
 'tokens': ['Pt',
  'with',
  'Type',
  '2',
  'Diabetes',
  'on',
  'Metformin.',
  'HbA1c',
  'today',
  '8.1%.',
  'No',
  'chest',
  'pain.',
  'Return',
  'in',
  '2',
  'weeks.'],
 'normalized_text': 'patient with type 2 diabetes on metformin. hba1c today 8.1%. no chest pain. return in 2 weeks.',
 'entities': [{'text': 'pt',
   'label': 'Diagnostic_procedure',
   'confidence': np.float32(0.97)},
  {'text': 'type 2', 'label': 'History', 'confidence': np.float32(0.71)},
  {'text': 'diabetes',
   'label': 'Disease_disorder',
   'confidence': np.float32(0.73)},
  {'text': 'met', 'label': 'Medication', 'confidence': np.float32(1.0)},
  {'text': '##form', 'label': 'Medication', 'confidence': np.float32(0.98)},
  {'text': 'h',
   'label': 'Diagnostic_procedure',
   'confidence': np.float32(0.59)},
  {'text': '##ba1c',
   'label': 'Diagnostic_procedure',

Final Summary

You built this workflow:

Clinical Note
↓
Tokenization + Normalization
↓
NER, find entities
↓
Context + Time understanding
↓
Classification, predict coding concepts
↓
Map to ICD/CPT candidates
↓
Final coded output

The most important idea:

The model should not jump directly from note to code.

A safer workflow is:

Text
↓
Evidence
↓
Clinical concept
↓
Candidate code
↓
Human review

This is better because the doctor or coder can see why each code was suggested.